# Task 04 — Exploratory Data Analysis & Statistical Insights

**Internship:** RABTECH Academy — Data Analytics & Business Intelligence  
**Dataset:** `clean_dataset.csv` (retail A/B test dataset)

## Objectives
- Compute descriptive statistics (mean, median, standard deviation, quartiles).
- Explore distributions using histograms and box plots.
- Build a correlation heatmap for numerical variables.
- Test three business hypotheses using statistical tests.
- Summarize five important findings in Markdown.

> Keep `clean_dataset.csv` in the same folder as this notebook before running it. This dataset is observational/simulated A/B-test-style data; findings describe this dataset and should not automatically be generalized to real customers.

## 1. Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

# Load the Task 03 cleaned dataset
df = pd.read_csv("clean_dataset.csv")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 2. Basic checks

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())
display(df.dtypes.to_frame("data_type"))


## 3. Descriptive statistics

In [ ]:
numeric_df = df.select_dtypes(include="number")
summary = numeric_df.describe().T
summary["median"] = numeric_df.median()
summary = summary[["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"]]
display(summary.round(2))


## 4. Histograms — distributions

In [ ]:
hist_cols = [c for c in [
    "age", "income_k", "prior_3mo_spend", "session_duration_sec",
    "pages_viewed", "order_value", "satisfaction_score"
] if c in df.columns]

for col in hist_cols:
    plt.figure(figsize=(7, 4))
    df[col].dropna().hist(bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 5. Box plots — possible outliers

In [ ]:
box_cols = [c for c in [
    "income_k", "prior_3mo_spend", "session_duration_sec",
    "pages_viewed", "order_value", "shipping_cost"
] if c in df.columns]

for col in box_cols:
    plt.figure(figsize=(7, 2.8))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f"Box plot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


## 6. Correlation heatmap

In [ ]:
corr_cols = [c for c in [
    "age", "income_k", "prior_3mo_visits", "prior_3mo_spend",
    "session_duration_sec", "pages_viewed", "cart_adds",
    "purchased", "order_value", "discount_applied",
    "shipping_cost", "returned", "satisfaction_score", "conversion_flag"
] if c in df.columns]

corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Correlation heatmap (Pearson)")
fig.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

display(corr.round(2))


## 7. Business hypotheses

We use a 5% significance level (`alpha = 0.05`). A p-value below 0.05 is evidence against the null hypothesis, but does not by itself prove causation or business importance.

**Hypothesis 1 — Conversion rate**
- H₀: Control and Discount10 have equal conversion rates.
- H₁: Their conversion rates differ.
- Test: two-proportion z-test (or chi-square test).

**Hypothesis 2 — Order value among purchasers**
- H₀: Mean order value is equal between Control and Discount10 among purchasers.
- H₁: The means differ.
- Test: Welch's independent-samples t-test.

**Hypothesis 3 — Satisfaction**
- H₀: Mean satisfaction score is equal between Control and Discount10.
- H₁: The means differ.
- Test: Welch's independent-samples t-test.

In [ ]:
alpha = 0.05
group_col = "experiment_group"

if group_col not in df.columns:
    raise ValueError("Expected column 'experiment_group' was not found.")

print("Experiment groups:", df[group_col].value_counts(dropna=False).to_dict())


### Hypothesis 1: Conversion rate

In [ ]:
# Uses conversion_flag if available; otherwise uses purchased.
outcome = "conversion_flag" if "conversion_flag" in df.columns else "purchased"
groups = df[group_col].dropna().unique().tolist()

if "Control" in groups and "Discount10" in groups:
    control = df.loc[df[group_col] == "Control", outcome].dropna()
    discount = df.loc[df[group_col] == "Discount10", outcome].dropna()

    x1, n1 = int(control.sum()), len(control)
    x2, n2 = int(discount.sum()), len(discount)
    p1, p2 = x1 / n1, x2 / n2
    pooled = (x1 + x2) / (n1 + n2)
    se = np.sqrt(pooled * (1 - pooled) * (1/n1 + 1/n2))
    z_stat = (p2 - p1) / se if se > 0 else 0
    p_value = 2 * stats.norm.sf(abs(z_stat))

    print(f"Outcome: {outcome}")
    print(f"Control conversion: {p1:.1%} ({x1}/{n1})")
    print(f"Discount10 conversion: {p2:.1%} ({x2}/{n2})")
    print(f"Difference (Discount10 - Control): {p2-p1:.1%}")
    print(f"z = {z_stat:.3f}, p-value = {p_value:.4f}")
    print("Decision:", "Reject H0" if p_value < alpha else "Fail to reject H0")
else:
    print("Control and/or Discount10 groups are missing. Review group names above.")


### Hypothesis 2: Mean order value among purchasers

In [ ]:
if {"purchased", "order_value"}.issubset(df.columns) and {"Control", "Discount10"}.issubset(set(df[group_col].dropna())):
    control_orders = df.loc[(df[group_col] == "Control") & (df["purchased"] == 1), "order_value"].dropna()
    discount_orders = df.loc[(df[group_col] == "Discount10") & (df["purchased"] == 1), "order_value"].dropna()

    if len(control_orders) > 1 and len(discount_orders) > 1:
        t_stat, p_value = stats.ttest_ind(discount_orders, control_orders, equal_var=False)
        print(f"Control mean order value: {control_orders.mean():.2f} (n={len(control_orders)})")
        print(f"Discount10 mean order value: {discount_orders.mean():.2f} (n={len(discount_orders)})")
        print(f"Difference (Discount10 - Control): {discount_orders.mean()-control_orders.mean():.2f}")
        print(f"Welch t = {t_stat:.3f}, p-value = {p_value:.4f}")
        print("Decision:", "Reject H0" if p_value < alpha else "Fail to reject H0")
    else:
        print("Not enough purchaser records in one or both groups.")
else:
    print("Required columns or experiment groups are missing.")


### Hypothesis 3: Mean satisfaction score

In [ ]:
if "satisfaction_score" in df.columns and {"Control", "Discount10"}.issubset(set(df[group_col].dropna())):
    control_sat = df.loc[df[group_col] == "Control", "satisfaction_score"].dropna()
    discount_sat = df.loc[df[group_col] == "Discount10", "satisfaction_score"].dropna()

    t_stat, p_value = stats.ttest_ind(discount_sat, control_sat, equal_var=False)
    print(f"Control mean satisfaction: {control_sat.mean():.2f} (n={len(control_sat)})")
    print(f"Discount10 mean satisfaction: {discount_sat.mean():.2f} (n={len(discount_sat)})")
    print(f"Difference (Discount10 - Control): {discount_sat.mean()-control_sat.mean():.2f}")
    print(f"Welch t = {t_stat:.3f}, p-value = {p_value:.4f}")
    print("Decision:", "Reject H0" if p_value < alpha else "Fail to reject H0")
else:
    print("Required satisfaction column or experiment groups are missing.")


## 8. Group-level business metrics

Compare conversion, average order value, and satisfaction by experiment group. Conversion is the mean of a 0/1 flag.

In [ ]:
metric = "conversion_flag" if "conversion_flag" in df.columns else "purchased"
agg_dict = {
    metric: "mean",
    "order_value": "mean",
    "satisfaction_score": "mean",
}
available_agg = {k: v for k, v in agg_dict.items() if k in df.columns}
group_summary = df.groupby(group_col).agg(available_agg).rename(columns={metric: "conversion_rate", "order_value": "mean_order_value", "satisfaction_score": "mean_satisfaction"})
display(group_summary.round(3))


## 9. Top 5 critical findings — Markdown commentary

After running all cells, use the outputs to write five findings in your own words. Suggested structure:

1. **Dataset profile:** State the number of rows, columns, and whether missing/duplicate rows remain.
2. **Customer behavior:** Describe the shape of the order-value or session-duration distribution and note any possible outliers.
3. **Relationships:** Mention the strongest meaningful correlations from the heatmap. Correlation does not imply causation.
4. **Conversion test:** Compare Control and Discount10 conversion rates and state whether the p-value is below 0.05.
5. **Business recommendation:** Based on the conversion, order-value, and satisfaction results, recommend whether the discount deserves further testing. Consider statistical significance and practical effect size.

> Replace this guidance with your actual observations after executing the notebook. Do not claim an effect is significant unless the p-value supports that conclusion.

## 10. Conclusion

Summarize what the EDA suggests, note any limitations, and state one next analysis you would perform. Remember that statistical significance does not automatically mean a result is commercially meaningful.